<a href="https://colab.research.google.com/github/venukumarpj/infosys-maritime-ai/blob/main/Maritime_AI_Auth_and_Pricing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install streamlit pyjwt pyotp nest-asyncio pyngrok

In [14]:
%%writefile app.py
import streamlit as st
import jwt
import re
import random
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import sqlite3
from datetime import datetime, timedelta
import os

# Read credentials from environment variables injected by the Colab launcher
EMAIL_ADDRESS = os.environ.get("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD", "")
JWT_SECRET = os.environ.get("JWT_SECRET", "super_secret_jwt_key_12345")

ADMIN_USER = "admin"
ADMIN_PASS = "AdminSecure789#"

def init_db():
    conn = sqlite3.connect('users.db')
    c = conn.cursor()
    c.execute('''
        CREATE TABLE IF NOT EXISTS users (
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password TEXT,
            sec_question TEXT,
            sec_answer TEXT
        )
    ''')
    conn.commit()
    conn.close()

init_db()

def add_user(username, email, password, sec_question, sec_answer):
    try:
        conn = sqlite3.connect('users.db')
        c = conn.cursor()
        c.execute('INSERT INTO users VALUES (?, ?, ?, ?, ?)',
                  (username, email, password, sec_question, sec_answer))
        conn.commit()
        conn.close()
        return True, "Signup successful!"
    except sqlite3.IntegrityError:
        return False, "Username or Email already exists."

def get_user_by_username(username):
    conn = sqlite3.connect('users.db')
    c = conn.cursor()
    c.execute('SELECT * FROM users WHERE username = ?', (username,))
    user = c.fetchone()
    conn.close()
    return user

def get_user_by_email(email):
    conn = sqlite3.connect('users.db')
    c = conn.cursor()
    c.execute('SELECT * FROM users WHERE email = ?', (email,))
    user = c.fetchone()
    conn.close()
    return user

def update_password(username_or_email, new_password):
    conn = sqlite3.connect('users.db')
    c = conn.cursor()
    c.execute('UPDATE users SET password = ? WHERE username = ? OR email = ?',
              (new_password, username_or_email, username_or_email))
    conn.commit()
    conn.close()

def get_all_users():
    conn = sqlite3.connect('users.db')
    c = conn.cursor()
    c.execute('SELECT username, email, sec_question FROM users')
    users = c.fetchall()
    conn.close()
    return users

# Email regex supporting numbers and multi-level custom domains (e.g. @saec.ac.in)
def validate_email(email):
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return re.match(pattern, email) is not None

def validate_password(password):
    if len(password) < 8:
        return False, "Password must be at least 8 characters long."
    if not re.search(r'[A-Z]', password):
        return False, "Password must contain at least one uppercase letter."
    if not re.search(r'[a-z]', password):
        return False, "Password must contain at least one lowercase letter."
    if not re.search(r'\d', password):
        return False, "Password must contain at least one number."
    if not re.search(r'[!@#$%^&*(),.?":{}|<>]', password):
        return False, "Password must contain at least one special symbol."
    return True, "Valid password"

def send_otp_email(recipient_email, otp):
    if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
        return False, "Missing EMAIL_ADDRESS or EMAIL_PASSWORD environment variables."
    try:
        msg = MIMEMultipart()
        msg['From'] = EMAIL_ADDRESS
        msg['To'] = recipient_email
        msg['Subject'] = "Password Reset OTP - Maritime Freight AI"
        body = f"Your One-Time Password (OTP) for password reset is: {otp}\nValid for 10 minutes."
        msg.attach(MIMEText(body, 'plain'))

        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        server.sendmail(EMAIL_ADDRESS, recipient_email, msg.as_string())
        server.quit()
        return True, "Success"
    except Exception as e:
        return False, str(e)

def create_token(username):
    payload = {
        'sub': username,
        'exp': datetime.utcnow() + timedelta(hours=2)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm='HS256')

def verify_token(token):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=['HS256'])
        return payload['sub']
    except jwt.ExpiredSignatureError:
        return None
    except jwt.InvalidTokenError:
        return None

# --- WIDE PAGE CONFIGURATION ---
st.set_page_config(page_title="Maritime Freight Pricing & Route Optimization", page_icon="🌊", layout="wide")

# --- DEEP OCEAN THEME STYLING ---
st.markdown("""
    <style>
    .stApp {
        background: linear-gradient(135deg, #0b1d3a 0%, #083358 50%, #0284c7 100%);
        color: #ffffff !important;
        font-family: 'Inter', sans-serif;
    }
    [data-testid="stSidebar"] {
        background-color: #061a33;
        border-right: 1px solid #0ea5e9;
    }
    [data-testid="stSidebar"] .stRadio label, [data-testid="stSidebar"] span, [data-testid="stSidebar"] p {
        color: #e0f2fe !important;
        font-weight: 500;
    }
    label, p, span, div, .stMarkdown {
        color: #ffffff !important;
    }
    h1, h2, h3 {
        color: #38bdf8 !important;
        font-weight: 700;
        letter-spacing: 0.5px;
    }
    input, select, textarea {
        background-color: #061a33 !important;
        color: #ffffff !important;
        -webkit-text-fill-color: #ffffff !important;
        border: 1px solid #0ea5e9 !important;
        border-radius: 8px !important;
    }
    [data-baseweb="input"] {
        background-color: #061a33 !important;
        border-radius: 8px !important;
        border: 1px solid #0ea5e9 !important;
    }
    [data-baseweb="input"] input {
        color: #ffffff !important;
        -webkit-text-fill-color: #ffffff !important;
    }
    .stButton button {
        background: linear-gradient(90deg, #f43f5e 0%, #fb7185 100%);
        color: #ffffff !important;
        border: none;
        border-radius: 8px;
        padding: 0.6rem 1rem;
        font-weight: 600;
        width: 100%;
        transition: all 0.3s ease;
        box-shadow: 0 4px 12px rgba(244, 63, 94, 0.4);
    }
    .stButton button:hover {
        background: linear-gradient(90deg, #e11d48 0%, #f43f5e 100%);
        box-shadow: 0 6px 18px rgba(244, 63, 94, 0.6);
    }
    </style>
""", unsafe_allow_html=True)

if 'token' not in st.session_state:
    st.session_state.token = None
if 'is_admin' not in st.session_state:
    st.session_state.is_admin = False
if 'otp_storage' not in st.session_state:
    st.session_state.otp_storage = {}

def main():
    st.sidebar.title("🌊 Ocean Freight AI")
    st.sidebar.markdown("---")
    st.sidebar.markdown("**Agentic AI for Maritime Freight Pricing & Route Optimization**")
    st.sidebar.markdown("---")

    logged_in_user = verify_token(st.session_state.token) if st.session_state.token else None

    if logged_in_user:
        choice = st.sidebar.radio("Navigation", ["Dashboard", "Logout"])
        if choice == "Dashboard":
            user_dashboard(logged_in_user)
        elif choice == "Logout":
            st.session_state.token = None
            st.session_state.is_admin = False
            st.rerun()
    elif st.session_state.is_admin:
        choice = st.sidebar.radio("Navigation", ["Admin Dashboard", "Logout"])
        if choice == "Admin Dashboard":
            admin_dashboard()
        elif choice == "Logout":
            st.session_state.is_admin = False
            st.rerun()
    else:
        choice = st.sidebar.radio("Navigation", ["Login", "Signup", "Forgot Password", "Admin Login"])
        if choice == "Login":
            login_page()
        elif choice == "Signup":
            signup_page()
        elif choice == "Forgot Password":
            forgot_password_page()
        elif choice == "Admin Login":
            admin_login_page()

def login_page():
    st.markdown("<h1 style='text-align: center;'>🌊 SAVE THE OCEANS & FREIGHT</h1>", unsafe_allow_html=True)
    st.markdown("<p style='text-align: center; color: #7dd3fc;'>Agentic AI-Powered Maritime Pricing & Route Optimization Portal</p>", unsafe_allow_html=True)
    st.markdown("<br>", unsafe_allow_html=True)

    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        with st.form("login_form"):
            st.markdown("### Secure Sign In")
            identifier = st.text_input("Username or Email *")
            password = st.text_input("Password *", type="password")
            submit = st.form_submit_button("SECURE LOGIN")
            if submit:
                if not identifier or not password:
                    st.error("All mandatory fields must be filled.")
                else:
                    user = get_user_by_username(identifier) or get_user_by_email(identifier)
                    if user and user[2] == password:
                        st.session_state.token = create_token(user[0])
                        st.success("Authentication successful! Loading workspace...")
                        st.rerun()
                    else:
                        st.error("Invalid username/email or password.")

def signup_page():
    st.markdown("<h1 style='text-align: center;'>⚓ Join the Movement</h1>", unsafe_allow_html=True)
    st.markdown("<p style='text-align: center; color: #7dd3fc;'>Register a new operator account for the platform.</p>", unsafe_allow_html=True)
    st.markdown("<br>", unsafe_allow_html=True)

    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        with st.form("signup_form"):
            st.markdown("### Operator Signup")
            username = st.text_input("Username *")
            email = st.text_input("Email * (e.g. 2324109@saec.ac.in)")
            password = st.text_input("Password *", type="password")
            confirm_password = st.text_input("Confirm Password *", type="password")
            security_questions = [
                "What was your first vessel name?",
                "What is your home port city?",
                "What was the name of your first maritime academy?",
                "What is your favorite ocean route?"
            ]
            sec_question = st.selectbox("Security Question *", security_questions)
            sec_answer = st.text_input("Security Answer *")
            submit = st.form_submit_button("REGISTER ACCOUNT")
            if submit:
                if not all([username, email, password, confirm_password, sec_answer]):
                    st.error("All fields are mandatory.")
                elif not validate_email(email):
                    st.error("Invalid email format. Please enter a valid email address.")
                else:
                    valid_pass, pass_msg = validate_password(password)
                    if not valid_pass:
                        st.error(pass_msg)
                    elif password != confirm_password:
                        st.error("Passwords do not match.")
                    else:
                        success, msg = add_user(username, email, password, sec_question, sec_answer)
                        if success:
                            st.success(msg + " Please proceed to Login.")
                        else:
                            st.error(msg)

def forgot_password_page():
    st.markdown("<h1 style='text-align: center;'>🌊 Freight Agent Recovery</h1>", unsafe_allow_html=True)
    st.markdown("<br>", unsafe_allow_html=True)

    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        if 'forgot_step' not in st.session_state:
            st.session_state.forgot_step = 'choose_method'

        recovery_method = st.radio("Choose Recovery Protocol", ["Security Question", "Email OTP"], key="recovery_method_selector")

        if recovery_method == "Security Question":
            if st.session_state.get('active_fp_tab') != 'sq':
                st.session_state.active_fp_tab = 'sq'
                st.session_state.forgot_step = 'sq_enter_user'

            if st.session_state.forgot_step == 'sq_enter_user':
                with st.form("sq_user_form"):
                    username_input = st.text_input("Enter Username *")
                    fetch_btn = st.form_submit_button("FETCH SECURITY QUESTION")
                    if fetch_btn:
                        user = get_user_by_username(username_input)
                        if user:
                            st.session_state['fp_username'] = username_input
                            st.session_state['fp_question'] = user[3]
                            st.session_state.forgot_step = 'sq_answer'
                            st.rerun()
                        else:
                            st.error("Username not found in system.")

            elif st.session_state.forgot_step == 'sq_answer':
                st.info(f"Recovering account for: **{st.session_state.get('fp_username')}**")
                with st.form("sq_reset_form"):
                    st.write(f"**Security Question:** {st.session_state.get('fp_question')}")
                    ans_input = st.text_input("Your Answer *")
                    new_p = st.text_input("New Password *", type="password")
                    conf_p = st.text_input("Confirm New Password *", type="password")
                    reset_btn = st.form_submit_button("RESET PASSWORD")
                    if reset_btn:
                        user_record = get_user_by_username(st.session_state['fp_username'])
                        if ans_input.strip().lower() != user_record[4].strip().lower():
                            st.error("Incorrect security answer.")
                        else:
                            valid_p, p_msg = validate_password(new_p)
                            if not valid_p:
                                st.error(p_msg)
                            elif new_p != conf_p:
                                st.error("Passwords do not match.")
                            else:
                                update_password(st.session_state['fp_username'], new_p)
                                st.success("Password successfully reset! Please proceed to Login.")
                                st.session_state.forgot_step = 'sq_enter_user'

        elif recovery_method == "Email OTP":
            if st.session_state.get('active_fp_tab') != 'otp':
                st.session_state.active_fp_tab = 'otp'
                st.session_state.forgot_step = 'otp_request'

            if st.session_state.forgot_step == 'otp_request':
                with st.form("otp_req_form"):
                    email_input = st.text_input("Registered Email *")
                    send_otp_btn = st.form_submit_button("SEND VERIFICATION OTP")
                    if send_otp_btn:
                        user = get_user_by_email(email_input)
                        if not user:
                            st.error("Email not found in system. Please register first.")
                        else:
                            otp = str(random.randint(100000, 999999))
                            st.session_state.otp_storage[email_input] = otp
                            success, err_msg = send_otp_email(email_input, otp)
                            if success:
                                st.success("OTP sent successfully to your email!")
                                st.session_state['fp_email'] = email_input
                                st.session_state.forgot_step = 'otp_verify'
                                st.rerun()
                            else:
                                st.error(f"Failed to send email: {err_msg}")

            elif st.session_state.forgot_step == 'otp_verify':
                st.info(f"OTP sent to: **{st.session_state.get('fp_email')}**")
                with st.form("otp_ver_form"):
                    entered_otp = st.text_input("Enter 6-digit OTP *")
                    new_p = st.text_input("New Password *", type="password")
                    conf_p = st.text_input("Confirm New Password *", type="password")
                    verify_reset_btn = st.form_submit_button("VERIFY & RESET PASSWORD")
                    if verify_reset_btn:
                        target_email = st.session_state['fp_email']
                        if st.session_state.otp_storage.get(target_email) == entered_otp:
                            valid_p, p_msg = validate_password(new_p)
                            if not valid_p:
                                st.error(p_msg)
                            elif new_p != conf_p:
                                st.error("Passwords do not match.")
                            else:
                                update_password(target_email, new_p)
                                st.success("Password reset successful! Please login.")
                                del st.session_state.otp_storage[target_email]
                                st.session_state.forgot_step = 'otp_request'
                        else:
                            st.error("Invalid or expired OTP.")

def admin_login_page():
    st.markdown("<h1 style='text-align: center;'>🛡️ System Admin Portal</h1>", unsafe_allow_html=True)
    st.markdown("<br>", unsafe_allow_html=True)

    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        with st.form("admin_form"):
            st.markdown("### Administrative Sign In")
            user = st.text_input("Admin Username *")
            pwd = st.text_input("Admin Password *", type="password")
            sub = st.form_submit_button("AUTHENTICATE ADMIN")
            if sub:
                if user == ADMIN_USER and pwd == ADMIN_PASS:
                    st.session_state.is_admin = True
                    st.success("Admin login successful!")
                    st.rerun()
                else:
                    st.error("Admin credentials incorrect.")

def admin_dashboard():
    st.title("🛡️ Admin Dashboard")
    st.markdown("### Registered Maritime Operators List")
    users = get_all_users()
    if users:
        user_data = [{"Username": u[0], "Email": u[1], "Security Question": u[2]} for u in users]
        st.table(user_data)
    else:
        st.info("No registered users found.")

def user_dashboard(username):
    st.title("🚢 Maritime Agent Workspace")
    st.success(f"Welcome back, Captain/Operator **{username}**!")
    st.markdown("You have successfully authenticated via JWT. Your agentic pricing and route optimization control center is active.")

if __name__ == '__main__':
    main()

Overwriting app.py


In [15]:
from pyngrok import ngrok
from google.colab import userdata
import subprocess
import time
import os

# 1. Load Colab secrets into environment variables so Streamlit can read them
try:
    os.environ['EMAIL_ADDRESS'] = userdata.get('EMAIL_ADDRESS')
    os.environ['EMAIL_PASSWORD'] = userdata.get('EMAIL_PASSWORD')
    os.environ['JWT_SECRET'] = userdata.get('JWT_SECRET')
    print("✅ Colab Secrets successfully injected into environment variables!")
except Exception as e:
    print(f"⚠️ Error loading secrets: {e}")

# 2. Terminate old background instances
subprocess.run(["pkill", "-9", "-f", "streamlit"])
subprocess.run(["pkill", "-9", "-f", "ngrok"])
ngrok.kill()
time.sleep(2)

# 3. Set Ngrok Token
try:
    ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
except Exception:
    pass

# 4. Start Streamlit in the background
get_ipython().system_raw("streamlit run app.py &")
time.sleep(3)

# 5. Connect to static URL
try:
    public_url = ngrok.connect(addr=8501, url="quail-shrubbery-wolf.ngrok-free.dev")
    print(f"🚢 App Live at: {public_url}")
except Exception:
    public_url = ngrok.connect(8501)
    print(f"🚢 App Live at: {public_url}")

✅ Colab Secrets successfully injected into environment variables!
🚢 App Live at: NgrokTunnel: "https://quail-shrubbery-wolf.ngrok-free.dev" -> "http://localhost:8501"
